# ANÁLISE DE ESTOQUE
Análise por: Victória Correa

In [1]:
## CARREGAMENTO

import pandas as pd
import numpy as np

arquivo = "dados.xlsx"

df_consumo = pd.read_excel(arquivo, sheet_name="Consumo")
df_estoque = pd.read_excel(arquivo, sheet_name="Estoque")
df_compras = pd.read_excel(arquivo, sheet_name="Compras")

print("Tabelas carregadas com sucesso!")

Tabelas carregadas com sucesso!


## Etapa 1 – Diagnóstico
### 1. Materiais com Risco de Ruptura

In [2]:
df_analise = df_estoque.copy()

df_analise['Diferenca_Para_Minimo'] = df_analise['Estoque Atual'] - df_analise['Estoque Mínimo']

# Separando os grupos de risco
df_critico = df_analise[df_analise['Estoque Atual'] < df_analise['Estoque Mínimo']]
df_igual = df_analise[df_analise['Estoque Atual'] == df_analise['Estoque Mínimo']]

# Alerta: quem está acima do mínimo, mas por muito pouco (ex: menos de 15 unidades de margem)
df_alerta = df_analise[(df_analise['Estoque Atual'] > df_analise['Estoque Mínimo']) & (df_analise['Diferenca_Para_Minimo'] <= 15)]

print("=== 1. RUPTURA CRÍTICA (Abaixo do Mínimo) ===")
if not df_critico.empty:
    display(df_critico[['Item', 'Estoque Atual', 'Estoque Mínimo', 'Diferenca_Para_Minimo']])
    print(f"Materiais críticos em risco de ruptura: {df_critico['Item'].tolist()}\n")
else:    
    print("Não há materiais críticos em risco de ruptura\n")    

print("=== 2. NO LIMITE (Igual ao Mínimo) ===")
if not df_igual.empty: 
    display(df_igual[['Item', 'Estoque Atual', 'Estoque Mínimo', 'Diferenca_Para_Minimo']])
    print(f"Materiais no limite: {df_igual['Item'].tolist()}\n")
else:    
    print("Não há materiais no limite de risco de ruptura\n")

print("=== 3. ALERTA (Quase Rompendo - Margem menor ou igual a 15 unidades) ===")
if not df_alerta.empty:
    display(df_alerta[['Item', 'Estoque Atual', 'Estoque Mínimo', 'Diferenca_Para_Minimo']]) 
    print(f"Materiais em alerta: {df_alerta['Item'].tolist()}")
else: 
    print("Não há materiais próximos de risco de ruptura")

=== 1. RUPTURA CRÍTICA (Abaixo do Mínimo) ===


,Item,Estoque Atual,Estoque Mínimo,Diferenca_Para_Minimo
14,A015,96,121,-25
16,A017,88,92,-4


Materiais críticos em risco de ruptura: ['A015', 'A017']

=== 2. NO LIMITE (Igual ao Mínimo) ===
Não há materiais no limite de risco de ruptura

=== 3. ALERTA (Quase Rompendo - Margem menor ou igual a 15 unidades) ===
Não há materiais próximos de risco de ruptura


### 2. Materiais com Excesso de Estoque

In [3]:
# Calcular a média de consumo mensal para cada item
df_media_consumo = df_consumo.groupby('Item')['Quantidade Consumida'].mean().reset_index()
df_media_consumo.columns = ['Item', 'Consumo_Medio_Mensal']
df_analise = pd.merge(df_analise, df_media_consumo, on='Item', how='left')

# Calcular a Cobertura de Estoque (Estoque Atual / Consumo Médio)
df_analise['Cobertura_Meses'] = (df_analise['Estoque Atual'] / df_analise['Consumo_Medio_Mensal']).round(1)

# Se o estoque atual for maior que o mínimo E durar mais de 2.0 meses no estoque
df_excesso_real = df_analise[
    (df_analise['Estoque Atual'] > df_analise['Estoque Mínimo']) & 
    (df_analise['Cobertura_Meses'] > 2.0)
]

print("=== EXCESSO REAL BASEADO EM COBERTURA DE CONSUMO (Mais de 2 meses de estoque) ===")
if not df_excesso_real.empty:
    display(df_excesso_real[['Item', 'Estoque Atual', 'Estoque Mínimo', 'Consumo_Medio_Mensal', 'Cobertura_Meses', 'Lote Mínimo']])
    print(f"Materiais com capital imobilizado (excesso): {df_excesso_real['Item'].tolist()}")
else:
    print("Nenhum material possui mais de 2 meses de cobertura.")

=== EXCESSO REAL BASEADO EM COBERTURA DE CONSUMO (Mais de 2 meses de estoque) ===


,Item,Estoque Atual,Estoque Mínimo,Consumo_Medio_Mensal,Cobertura_Meses,Lote Mínimo
1,A002,217,111,92.083333,2.4,100
3,A004,169,122,77.583333,2.2,200
10,A011,280,123,79.083333,3.5,200
15,A016,253,75,116.750000,2.2,100
17,A018,299,69,106.250000,2.8,100


Materiais com capital imobilizado (excesso): ['A002', 'A004', 'A011', 'A016', 'A018']


## 3. Impacto Financeiro

In [4]:
# Impacto da Ruptura
pecas_faltantes = df_critico['Estoque Mínimo'] - df_critico['Estoque Atual']
total_faltando = pecas_faltantes.sum()

# Impacto do Excesso 
pecas_excesso = df_excesso_real['Estoque Atual'] - (df_excesso_real['Consumo_Medio_Mensal'] * 2)
total_excesso = pecas_excesso.sum()

# Resultado
print("=== ANÁLISE DE IMPACTO POR VOLUME DE PEÇAS ===")
print(f"Total de peças que FALTAM para atingir o mínimo: {total_faltando:.0f} unidades")
print(f"Total de peças em EXCESSO (acima de 2 meses):     {total_excesso:.0f} unidades")

=== ANÁLISE DE IMPACTO POR VOLUME DE PEÇAS ===
Total de peças que FALTAM para atingir o mínimo: 29 unidades
Total de peças em EXCESSO (acima de 2 meses):     274 unidades


Conclusão do Diagnóstico de Impacto Financeiro:
O maior impacto financeiro quantificável está no Excesso de Estoque, que acumula 274 unidades acima do limite estratégico de 2 meses de consumo, gerando um alto volume de capital imobilizado (dinheiro parado no galpão).

Por outro lado, embora o impacto do déficit seja menor em volume (29 unidades), ele representa o maior risco operacional para o negócio: o risco de ruptura iminente, que pode paralisar o processo produtivo e gerar prejuízos por perda de faturamento.

## 3. Fornecedores com Pior Desempenho

In [5]:
df_compras['Data Compra'] = pd.to_datetime(df_compras['Data Compra'], dayfirst=True)
df_compras['Data Entrega'] = pd.to_datetime(df_compras['Data Entrega'], dayfirst=True)

# Calcular quantos dias levou para entregar (Lead Time Real)
df_compras['Lead_Time_Real'] = (df_compras['Data Entrega'] - df_compras['Data Compra']).dt.days

# Trazer o prazo prometido (Lead Time) da tabela de estoque para a de compras
df_desempenho = pd.merge(df_compras, df_estoque[['Item', 'Lead Time (dias)']], on='Item', how='left')

# Calcular os dias de atraso (Real - Prometido)
df_desempenho['Dias_Atraso'] = df_desempenho['Lead_Time_Real'] - df_desempenho['Lead Time (dias)']

df_atrasos = df_desempenho[df_desempenho['Dias_Atraso'] > 0]

# Resultados
if not df_atrasos.empty:
    # Média de dias de atraso
    df_media_atrasos = df_atrasos.groupby('Item')['Dias_Atraso'].mean().reset_index()
    df_media_atrasos['Dias_Atraso'] = df_media_atrasos['Dias_Atraso'].round(1)
    df_media_atrasos.columns = ['Item', 'Media_Dias_Atraso_Por_Pedido']
    
    ranking_piores_media = df_media_atrasos.sort_values('Media_Dias_Atraso_Por_Pedido', ascending=False)
    
    print("\n=== MATERIAIS COM MAIOR MÉDIA DE ATRASO NA ENTREGA ===")
    display(ranking_piores_media)
    
else:
    print("Nenhum material sofreu atrasos nas entregas no histórico!")


=== MATERIAIS COM MAIOR MÉDIA DE ATRASO NA ENTREGA ===


,Item,Media_Dias_Atraso_Por_Pedido
0,A001,16.7
3,A005,12.7
15,A020,9.8
8,A013,9.8
12,A017,9.0
6,A011,8.0
4,A008,8.0
7,A012,7.0
5,A010,6.0
9,A014,4.7


## Etapa 2 – Análise Estratégica do Estoque

## 1. Aumento Estoque x Melhoria Atendimento

In [6]:
# Criar a coluna de Cobertura em Meses (Estoque Atual dividido pelo Consumo Mensal)
df_analise['Cobertura_Meses'] = df_analise['Estoque Atual'] / df_analise['Consumo_Medio_Mensal']

# Contar quantos itens estão desregulados usando o consumo como base:
# Risco: Estoque dura menos de 1 mês de consumo
itens_risco_consumo = df_analise[df_analise['Cobertura_Meses'] < 1.0].shape[0]

# Excesso: Estoque dura mais de 2 meses de consumo 
itens_excesso_consumo = df_analise[df_analise['Cobertura_Meses'] > 2.0].shape[0]

print("=== ANÁLISE DE ATENDIMENTO BASEADA NO CONSUMO ===")
print(f"Materiais com estoque CRÍTICO (duram menos de 1 mês de consumo): {itens_risco_consumo} itens")
print(f"Materiais com estoque EM EXCESSO (duram mais de 2 meses de consumo): {itens_excesso_consumo} itens")

=== ANÁLISE DE ATENDIMENTO BASEADA NO CONSUMO ===
Materiais com estoque CRÍTICO (duram menos de 1 mês de consumo): 6 itens
Materiais com estoque EM EXCESSO (duram mais de 2 meses de consumo): 6 itens


**Conclusâo:** Não, o aumento do estoque não está gerando melhoria no atendimento devido a um **desbalanceamento**. Os dados mostram que o capital pode ser melhor investido: temos itens com excesso parado (mais de 60 dias de cobertura), enquanto materiais de alta demanda operam na zona de risco (menos de 30 dias de cobertura), mantendo o risco de parada de fábrica ativo.


### 2. Os 10 Materiais Mais Críticos

In [7]:
# Ordenar a tabela inteira df_analise pela menor cobertura de meses
ranking_criticidade = df_analise.sort_values('Cobertura_Meses', ascending=True).head(10)

print("=== MATERIAIS MAIS CRÍTICOS (ORDENADOS POR MENOR COBERTURA) ===")
display(ranking_criticidade[['Item', 'Estoque Atual', 'Estoque Mínimo', 'Diferenca_Para_Minimo', 'Cobertura_Meses']])

=== MATERIAIS MAIS CRÍTICOS (ORDENADOS POR MENOR COBERTURA) ===


,Item,Estoque Atual,Estoque Mínimo,Diferenca_Para_Minimo,Cobertura_Meses
16,A017,88,92,-4,0.522255
7,A008,107,60,47,0.567388
14,A015,96,121,-25,0.693976
0,A001,158,106,52,0.906310
18,A019,149,93,56,0.927386
9,A010,166,60,106,0.984676
6,A007,186,71,115,1.063364
13,A014,234,88,146,1.192357
11,A012,142,126,16,1.244704
2,A003,163,89,74,1.260309


**Como cheguei a esse resultado (Critério):**
Em vez de olhar só para a quantidade de caixas no galpão, eu cruzei o estoque atual com o **consumo médio** de cada item. Isso me permitiu calcular a "Cobertura em Meses", que basicamente me diz: *"Se a empresa não comprar mais nada hoje, por quantos meses a fábrica consegue rodar antes de zerar tudo?"* Ordenei a lista deixando no topo quem tem menos tempo de sobrevivência.

**O que os dados me mostraram:**
* **Falta Real (Ruptura):** Os itens **A017** e **A015** são os únicos que já cruzaram a linha vermelha e estão operando abaixo do estoque mínimo estabelecido.
* **O Perigo Escondido (Alerta Máximo):** O item **A008** parece seguro porque está com 47 unidades acima do mínimo. Porém, como a fábrica consome esse material muito rápido, notei que todo o estoque atual dele só dura **17 dias** (0.56 meses). 

**Minha Conclusão:** Focar apenas no "Estoque Mínimo" antigo da empresa cria uma falsa sensação de segurança. Se o fornecedor do A008 atrasar duas semanas, a produção para, mesmo ele estando teoricamente no "verde". Por isso, usei o consumo como o meu melhor termômetro para mapear o risco real.

### 3. Atraso de Fornecedores x Rupturas

In [8]:
df_relacao = df_analise.merge(df_media_atrasos, on='Item', how='left')
df_relacao['Media_Dias_Atraso_Por_Pedido'] = df_relacao['Media_Dias_Atraso_Por_Pedido'].fillna(0)

# Calcular a média de atraso para o grupo de RISCO vs grupo SEGURO
filtro_criticos = df_relacao[df_relacao['Cobertura_Meses'] < 1.0]
filtro_seguros = df_relacao[df_relacao['Cobertura_Meses'] >= 1.0]

media_atraso_criticos = filtro_criticos['Media_Dias_Atraso_Por_Pedido'].mean()
media_atraso_seguros = filtro_seguros['Media_Dias_Atraso_Por_Pedido'].mean()

print("=== COMPARAÇÃO DO ATRASO DOS FORNECEDORES ===")
print(f"Média de atraso dos fornecedores para itens em RISCO CRÍTICO: {media_atraso_criticos:.1f} dias")
print(f"Média de atraso dos fornecedores para itens em ESTOQUE SEGURO: {media_atraso_seguros:.1f} dias")

=== COMPARAÇÃO DO ATRASO DOS FORNECEDORES ===
Média de atraso dos fornecedores para itens em RISCO CRÍTICO: 7.0 dias
Média de atraso dos fornecedores para itens em ESTOQUE SEGURO: 4.7 dias


**Minha Análise dos Dados:**
Ao cruzar o histórico de cumprimento de prazos dos fornecedores com a situação atual, identifiquei uma correlação direta entre a falta de pontualidade na entrega e o risco de desabastecimento na fábrica. 

Para comprovar isso, dividi os materiais em dois grupos e calculei a média de atraso de cada um:
* **Itens Críticos (menos de 1 mês de cobertura):** Os fornecedores desses materiais atrasam, em média, **7.0 dias** para entregar.
* **Itens Seguros (mais de 1 mês de cobertura):** Os fornecedores desses materiais apresentam uma média de atraso menor, de **4.7 dias**.

**Minha Conclusão:**
Embora exista uma diferença, eu notei que ela é relativamente pequena (apenas 2,3 dias de variação entre o grupo crítico e o saudável). Isso me leva a concluir que o atraso dos fornecedores **não é o único e nem o principal motivo** para os materiais estarem quebrando estoque. 

Para mim, esse resultado reforça o que descobri na questão anterior: o problema real está no desbalanceamento do planejamento interno. Mesmo que o fornecedor atrase pouco, se o nosso Estoque Mínimo estiver calculado abaixo do que o consumo real pede (como o caso do item A008), o material vai entrar em risco de qualquer forma.

### 3. Revisão Imediata de Parâmetros

In [9]:
# Definir os critérios de urgência para revisão:
# Cobertura menor que 1 mes (zona de risco)
# Média de atraso do fornecedor acima de 5 dias

filtro_revisao = df_relacao[
    (df_relacao['Cobertura_Meses'] < 1) & 
    (df_relacao['Media_Dias_Atraso_Por_Pedido'] > 5.0)
]

# Ordenar pelos casos mais graves (menor cobertura e maior atraso)
ranking_revisao = filtro_revisao.sort_values(
    by=['Cobertura_Meses', 'Media_Dias_Atraso_Por_Pedido'], 
    ascending=[True, False]
)

print("=== MATERIAIS PARA REVISÃO IMEDIATA DE PARÂMETROS ===")
display(ranking_revisao[[
    'Item', 'Estoque Atual', 'Estoque Mínimo', 
    'Consumo_Medio_Mensal', 'Cobertura_Meses', 'Media_Dias_Atraso_Por_Pedido'
]])

=== MATERIAIS PARA REVISÃO IMEDIATA DE PARÂMETROS ===


,Item,Estoque Atual,Estoque Mínimo,Consumo_Medio_Mensal,Cobertura_Meses,Media_Dias_Atraso_Por_Pedido
16,A017,88,92,168.500000,0.522255,9.0
7,A008,107,60,188.583333,0.567388,8.0
0,A001,158,106,174.333333,0.906310,16.7
9,A010,166,60,168.583333,0.984676,6.0


**Estratégia de Identificação:**
Para apontar com precisão quais materiais precisam de uma revisão urgente, cruzei os 4 pilares solicitados (`Lead Time`, `Consumo`, `Estoque de Segurança` e `Cobertura`) e busquei os itens que estão no pior dos mundos: **alta velocidade de consumo (baixa cobertura) combinada com um histórico de fornecedores que frequentemente atrasam**.

Os materiais listados na tabela acima são os que exigem ação imediata.

**Justificativa para a Revisão dos Parâmetros:**

1. **Lead Time e Estoque de Segurança:** Os dados mostram que o tempo de resposta real dos fornecedores desses itens está desalinhado com o que está cadastrado no sistema. Como o fornecedor atrasa, o Estoque de Segurança atual (que deveria proteger a operação) é engolido muito rápido e não dá conta. O Estoque de Segurança precisa ser recalculado para cima, considerando o desvio padrão desse atraso.
2. **Consumo e Cobertura:** Itens como o **A008** possuem um consumo tão agressivo que qualquer oscilação de 5 dias na entrega destrói a cobertura de estoque. 

**Recomendação Prática:**
Defendo que a equipe de planejamento aumente o fator de segurança do Lead Time desses materiais específicos e recalcule o lote mínimo de compra com base no consumo atualizado, evitando que fiquemos dependentes de uma pontualidade que o fornecedor historicamente não entrega.

In [10]:
# Exportar a tabela consolidada para um arquivo CSV na mesma pasta do projeto
df_relacao.to_csv('tabela_consolidada_dashboard.csv', index=False, sep=';', encoding='utf-8-sig')

print("Tabela salva com sucesso como 'tabela_consolidada_dashboard.csv'!")

Tabela salva com sucesso como 'tabela_consolidada_dashboard.csv'!
